# UPAR 2027 Track 2: frozen AIMv2-3B with a pooled attribute head

Run from top to bottom on Kaggle with **GPU and Internet enabled**. The notebook downloads only the official challenge data, loads the generic `timm/aimv2_3b_patch14_448.apple_pt` weights (10.9 GB, not gated), extracts frozen image features, trains a 40-attribute head, selects by validation retrieval mADM, and writes `/kaggle/working/upar_aimv2_3b_submission.zip`.

AIMv2-3B/14 was pretrained by Apple with a multimodal autoregressive objective on generic image-text pairs. The paper reports 89.5% ImageNet-1k top-1 for this checkpoint with a frozen trunk; no pedestrian or PAR data was used for the weights. They are released under Apple's research licence (`apple-ascl` on the `timm` repository), so check its terms before relying on it for the challenge.

With 2.7 billion parameters the trunk is too large to fine-tune on a Kaggle GPU, so it stays frozen, as in the DINOv3 notebook. The 448 x 448 position embedding is resampled to **448 x 224** so that pedestrian crops keep their aspect ratio.

**This is a heavy run.** Feature extraction over 131,076 images is likely to take several hours on a T4. The submission ZIP holds the trunk in half precision and is about **5.5 GB**; confirm that Codabench accepts an upload of that size and that the model fits the evaluation time limit before spending the GPU hours. The ZIP needs neither `timm` nor network access at inference.


## 1. Download the official challenge training and validation data

These data cells follow the existing ConvNeXt Tiny Kaggle notebook. They may be skipped if `/kaggle/working/upar_2027/data` is already complete.


In [ ]:
%pip -q install gdown tqdm requests timm

In [ ]:
import csv
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

if not (REPO / 'peta_file_mapping.txt').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
for split in ('train', 'val'):
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / 'data' / 'annotations' / f'{split}.csv', destination)
print('Data directory:', DATA)

In [ ]:
# The annotation paths determine exactly which images must be available.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        for row in list(csv.reader(handle))[1:]:
            relative = Path(row[0])
            needed[relative.parts[0]].add(relative)

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()  # Remove an interrupted download before retrying.
    return path.exists()

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}')
    print(f'{source}: {len(needed[source]):,} annotated images available')

print({source: len(paths) for source, paths in needed.items()})

## Market1501

In [ ]:
market_archive = DATA / 'market_1501.zip'
market_dir = DATA / 'Market1501'
if missing('Market1501'):
    if not check_zip(market_archive):
        result = gdown.download(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            output=str(market_archive), quiet=False, use_cookies=False, fuzzy=True
        )
        if not result or not zipfile.is_zipfile(market_archive):
            raise RuntimeError('Market1501 download failed or is not a ZIP file')
    with zipfile.ZipFile(market_archive) as archive:
        archive.extractall(DATA)
    extracted = DATA / 'Market-1501-v15.09.15'
    if extracted.exists():
        shutil.copytree(extracted, market_dir, dirs_exist_ok=True)
        shutil.rmtree(extracted)
require_complete('Market1501')
market_archive.unlink(missing_ok=True)

## PA-100K

In [ ]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
if missing('PA100k'):
    if not check_zip(pa_archive):
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
            output=str(pa_dir), quiet=False, use_cookies=False
        )
        if not zipfile.is_zipfile(pa_archive):
            raise RuntimeError('PA-100K data.zip was not downloaded')
    with zipfile.ZipFile(pa_archive) as archive:
        archive.extractall(pa_dir)
require_complete('PA100k')
pa_archive.unlink(missing_ok=True)

## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [ ]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
if missing('PETA'):
    if not check_zip(peta_archive):
        url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
        temporary = peta_archive.with_suffix('.zip.part')
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
                        bar.update(len(chunk))
        temporary.replace(peta_archive)
        if not zipfile.is_zipfile(peta_archive):
            raise RuntimeError('PETA download is not a ZIP file')
    with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
        mapping = list(csv.reader(handle))
    with zipfile.ZipFile(peta_archive) as archive:
        members = set(archive.namelist())
        for source, destination in tqdm(mapping, desc='Extracting PETA images'):
            target = DATA / destination
            if target.is_file():
                continue
            member = source.removeprefix('PETA/')
            if member not in members:
                raise FileNotFoundError(f'PETA archive is missing {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as original, target.open('wb') as output:
                shutil.copyfileobj(original, output)
require_complete('PETA')
peta_archive.unlink(missing_ok=True)

In [ ]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

## 2. Extract AIMv2 features and train the attribute head

Each image is described by the mean of all patch tokens, the mean of the upper half of the patch grid, and the mean of the lower half, giving a 9,216-dimensional feature. The head is a LayerNorm followed by one linear layer, trained with unweighted binary cross-entropy. A fixed set of 256 validation queries selects the checkpoint by mADM, and training stops after 3 checks without improvement.

The expensive feature extraction runs in half precision and is cached under `/kaggle/working` (about 2.4 GB); a restarted run reuses those files. Reduce `FEATURE_BATCH_SIZE` if the GPU runs out of memory. `IMAGE_SIZE = (336, 168)` roughly halves the extraction time. Both sides must be multiples of 14.


In [ ]:
"""Kaggle training and offline submission for UPAR 2027 Track 2.

AIMv2-3B experiment: frozen, generic pretrained AIMv2-3B/14 patch features are
pooled over the whole image and its upper and lower halves, and a 40-attribute
head is trained on the cached features. The checkpoint and the ranking distance
are selected by validation retrieval mADM.
"""

import csv
import importlib.util
import json
import os
import random
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


DATA_ROOT = Path(os.environ.get("UPAR_DATA_ROOT", "/kaggle/working/upar_2027/data"))
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
MODEL_ID = "aimv2_3b_patch14_448.apple_pt"  # Generic image-text pretraining; no PAR weights.
USE_PRETRAINED_WEIGHTS = True  # Kaggle training only; no runtime download.
IMAGE_SIZE = (448, 224)  # height, width; both must be multiples of the 14-pixel patch.
FEATURE_BATCH_SIZE = 16  # Reduce to 8 if the Kaggle GPU runs out of memory.
HEAD_BATCH_SIZE = 2048
HEAD_EPOCHS = 60
HEAD_LR = 1e-3
PATIENCE = 3  # Validation checks without improvement before stopping.
NUM_WORKERS = 2
SELECTION_QUERIES = 256
SEED = 42


def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attribute names in {path}")
        paths, labels = [], []
        for line, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{line}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.uint8)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path}: labels must be binary")
    return names, paths, labels


def image_transform(mean, std):
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


class ImageDataset(Dataset):
    def __init__(self, root, paths, transform):
        self.root, self.paths, self.transform = root, paths, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


class PortableRmsNorm(nn.Module):
    """RMSNorm from basic operations, computed in float32.

    The submission runs in half precision, where squaring large activations can
    overflow, and the traced graph must not depend on a newer PyTorch operator.
    """

    def __init__(self, norm):
        super().__init__()
        self.weight, self.eps = norm.weight, norm.eps

    def forward(self, tokens):
        wide = tokens.float()
        scale = wide.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (wide * scale).type_as(tokens) * self.weight


def replace_rms_norms(module):
    for name, child in module.named_children():
        if type(child).__name__ == "RmsNorm":
            setattr(module, name, PortableRmsNorm(child))
        else:
            replace_rms_norms(child)


class RegionPooledEncoder(nn.Module):
    """Mean of all patch tokens, of the upper half, and of the lower half."""

    def __init__(self, trunk):
        super().__init__()
        if trunk.num_prefix_tokens:
            raise ValueError("Expected a trunk without class or register tokens")
        self.trunk = trunk
        self.rows, self.columns = trunk.patch_embed.grid_size

    def forward(self, pixels):
        tokens = self.trunk.forward_features(pixels)
        grid = tokens.reshape(tokens.shape[0], self.rows, self.columns, tokens.shape[-1])
        half = self.rows // 2
        return torch.cat([tokens.mean(1), grid[:, :half].mean((1, 2)), grid[:, half:].mean((1, 2))], dim=-1)


class AttributeHead(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.norm = nn.LayerNorm(feature_dim)
        self.linear = nn.Linear(feature_dim, 40)

    def forward(self, features):
        return self.linear(self.norm(features))


class SubmissionModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder, self.head = encoder, head

    def forward(self, pixels):
        return self.head(self.encoder(pixels))


@torch.inference_mode()
def extract_features(encoder, paths, transform, cache_path, device):
    if cache_path.is_file():
        cached = torch.load(cache_path, map_location="cpu", weights_only=False)
        if (cached.get("paths") == paths and cached.get("model_id") == MODEL_ID
                and tuple(cached.get("image_size", ())) == tuple(IMAGE_SIZE)):
            print(f"Reusing {cache_path}")
            return cached["features"]
    loader = DataLoader(ImageDataset(DATA_ROOT, paths, transform), batch_size=FEATURE_BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
    dtype = next(encoder.parameters()).dtype
    pieces = []
    for batch_number, pixels in enumerate(loader, 1):
        features = encoder(pixels.to(device, dtype=dtype, non_blocking=True))
        if not torch.isfinite(features).all():
            raise ValueError("Nonfinite image features; half precision overflowed")
        pieces.append(features.cpu().half())
        if batch_number % 100 == 0:
            print(f"Extracted {min(batch_number * FEATURE_BATCH_SIZE, len(paths)):,}/{len(paths):,}", flush=True)
    features = torch.cat(pieces)
    torch.save({"features": features, "paths": paths, "model_id": MODEL_ID,
                "image_size": tuple(IMAGE_SIZE)}, cache_path)
    return features


@torch.inference_mode()
def predict_logits(head, features, device):
    head.eval()
    return np.concatenate([
        head(batch.float().to(device)).float().cpu().numpy()
        for batch in features.split(HEAD_BATCH_SIZE)
    ])


def distances_for(logits, queries, method):
    if method == "nll":
        return np.logaddexp(0, logits).sum(axis=1)[None, :] - queries @ logits.T
    if method == "l1":
        probabilities = 1 / (1 + np.exp(-np.clip(logits, -30, 30)))
        return probabilities.sum(axis=1)[None, :] + queries @ (1 - 2 * probabilities).T
    raise ValueError(method)


def retrieval_metrics(logits, labels, queries, method="nll"):
    """Challenge mADM, mAP, Rank-1/5/10, mINP for a validation gallery."""
    labels = labels.astype(np.float32)
    queries = queries.astype(np.float32)
    gallery_sum = labels.sum(axis=1)
    totals = np.zeros(6, dtype=np.float64)
    for start in range(0, len(queries), 16):
        query_batch = queries[start:start + 16]
        distances = distances_for(logits, query_batch, method)
        agreement = 40 - gallery_sum[None, :] - query_batch.sum(axis=1)[:, None] + 2 * query_batch @ labels.T
        for row in range(len(query_batch)):
            order = np.argsort(distances[row], kind="stable")
            ranks = np.flatnonzero(agreement[row, order] == 40) + 1
            if not len(ranks):
                raise ValueError("Validation query has no exact gallery match")
            mean_agreement = agreement[row].mean()
            normalized = np.maximum(0, (agreement[row, order] - mean_agreement) / (40 - mean_agreement))
            prefix = np.cumsum(normalized)
            totals += [np.mean(prefix[ranks - 1] / ranks),
                       np.mean(np.arange(1, len(ranks) + 1) / ranks),
                       ranks[0] <= 1, ranks[0] <= 5, ranks[0] <= 10,
                       len(ranks) / ranks[-1]]
    keys = ("mADM", "mAP", "Rank-1", "Rank-5", "Rank-10", "mINP")
    return dict(zip(keys, (totals / len(queries)).tolist()))


def fit_head(train_features, train_labels, val_features, val_labels, device):
    head = AttributeHead(train_features.shape[1]).to(device)
    loader = DataLoader(TensorDataset(train_features, torch.from_numpy(train_labels.astype(np.float32))),
                        batch_size=HEAD_BATCH_SIZE, shuffle=True, pin_memory=device.type == "cuda")
    queries = np.unique(val_labels, axis=0).astype(np.float32)
    rng = np.random.default_rng(SEED)
    selected = queries[rng.choice(len(queries), min(SELECTION_QUERIES, len(queries)), replace=False)]
    optimizer = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=0.01)
    checkpoint_path = OUTPUT_ROOT / "upar_aimv2_3b_head.pth"
    best_score, stale = -1.0, 0
    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        running = 0.0
        for features, targets in loader:
            features = features.float().to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(head(features), targets)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(features)
        if epoch == 1 or epoch % 5 == 0 or epoch == HEAD_EPOCHS:
            logits = predict_logits(head, val_features, device)
            metrics = retrieval_metrics(logits, val_labels, selected, "nll")
            print(f"Epoch {epoch:02d}: BCE={running / len(train_labels):.4f}; selection={metrics}", flush=True)
            if metrics["mADM"] > best_score:
                best_score, stale = metrics["mADM"], 0
                torch.save({"head": {k: v.detach().cpu() for k, v in head.state_dict().items()},
                            "epoch": epoch, "selection_mADM": best_score}, checkpoint_path)
            else:
                stale += 1
                if stale >= PATIENCE:
                    print(f"Stopping at epoch {epoch}: no improvement in {PATIENCE} checks")
                    break
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    head.load_state_dict(checkpoint["head"])
    print(f"Restored epoch {checkpoint['epoch']} with selection mADM={best_score:.4f}")
    logits = predict_logits(head, val_features, device)
    candidates = {name: retrieval_metrics(logits, val_labels, selected, name)
                  for name in ("nll", "l1")}
    score_method = max(candidates, key=lambda name: candidates[name]["mADM"])
    print("Selection scoring comparison:", candidates)
    print("Chosen score:", score_method)
    print("Full validation metrics:", retrieval_metrics(logits, val_labels, queries, score_method))
    return head.cpu().eval(), score_method


RUN_PY = '''"""Offline AIMv2-3B attribute retrieval for UPAR 2027 Track 2."""
from pathlib import Path
import json

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

HERE = Path(__file__).resolve().parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = None
CONFIG = None
TRANSFORM = None
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
BATCH_SIZE = 16


def load_model():
    global MODEL, CONFIG, TRANSFORM
    with (HERE / "assets/config.json").open() as handle:
        CONFIG = json.load(handle)
    TRANSFORM = transforms.Compose([
        transforms.Resize(tuple(CONFIG["image_size"]), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ])
    # The weights are stored in half precision; a CPU fallback widens them again.
    MODEL = torch.jit.load(str(HERE / "assets/model.pt"), map_location=DEVICE).eval()
    if DTYPE == torch.float32:
        MODEL.float()


def rank_gallery(sample):
    if MODEL is None:
        load_model()
    names = sample["attribute_names"]
    indices = [CONFIG["attribute_names"].index(name) for name in names]
    queries = np.asarray(sample["queries"], dtype=np.float32)
    if queries.ndim != 2 or queries.shape[1] != len(indices):
        raise ValueError("Expected a Q x 40 query matrix")
    gallery = sample["gallery"]
    distances = np.empty((len(queries), len(gallery)), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(gallery), BATCH_SIZE):
            batch = gallery[start:start + BATCH_SIZE]
            pixels = []
            for item in batch:
                with Image.open(item["image_path"]) as image:
                    pixels.append(TRANSFORM(image.convert("RGB")))
            pixels = torch.stack(pixels).to(DEVICE, dtype=DTYPE)
            logits = MODEL(pixels).float().cpu().numpy()[:, indices]
            if CONFIG["score_method"] == "nll":
                block = np.logaddexp(0, logits).sum(axis=1)[None, :] - queries @ logits.T
            else:
                probs = 1 / (1 + np.exp(-np.clip(logits, -30, 30)))
                block = probs.sum(axis=1)[None, :] + queries @ (1 - 2 * probs).T
            distances[:, start:start + len(batch)] = block
            if (start // BATCH_SIZE + 1) % 100 == 0:
                print(f"Ranked {start + len(batch):,}/{len(gallery):,} images", flush=True)
    if not np.isfinite(distances).all():
        raise ValueError("Nonfinite distances")
    return {"distances": distances}
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls rank_gallery(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(encoder, head, names, score_method, mean, std, transform, example_path, example_query):
    folder = OUTPUT_ROOT / "upar_aimv2_3b_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    model = SubmissionModel(encoder.float().cpu().eval(), head.cpu().eval())
    # Plain attention keeps the traced graph loadable by the runner's older PyTorch.
    for module in model.modules():
        if hasattr(module, "fused_attn"):
            module.fused_attn = False
    with Image.open(example_path) as image:
        example = transform(image.convert("RGB")).unsqueeze(0)
    print("Tracing on the CPU in float32; this takes several minutes", flush=True)
    with torch.inference_mode():
        traced = torch.jit.trace(model, example, check_trace=False)
        batch = example.repeat(2, 1, 1, 1)
        reference = model(batch)
        if not torch.allclose(reference, traced(batch), atol=1e-4, rtol=1e-4):
            raise ValueError("TorchScript export changed model predictions")
        traced.half()  # Halves the archive; the full-precision export would be about 11 GB.
        traced.save(str(assets / "model.pt"))
    del traced, model
    (assets / "config.json").write_text(json.dumps({
        "attribute_names": names, "score_method": score_method,
        "image_size": list(IMAGE_SIZE), "mean": list(mean), "std": list(std),
        "model_id": MODEL_ID,
    }, indent=2))
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    archive_path = OUTPUT_ROOT / "upar_aimv2_3b_submission.zip"
    expected = {"run.py", "metadata.yaml", "assets/model.pt", "assets/config.json"}
    # Stored, not deflated: floating-point weights barely compress.
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("aimv2_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            query = example_query.astype(np.float32)[None, :]
            result = module.rank_gallery({
                "attribute_names": names,
                "queries": query.astype(int).tolist(),
                "gallery": [{"index": 0, "image_path": str(example_path)}],
            })
            distances = np.asarray(result["distances"])
            assert distances.shape == (1, 1) and np.isfinite(distances).all()
            expected_distance = distances_for(reference[:1].numpy(), query, score_method)
            print(f"Packaged inference smoke test passed; distance {distances[0, 0]:.4f} "
                  f"against {expected_distance[0, 0]:.4f} in float32")
    print(f"Upload this ZIP to Codabench: {archive_path} ({archive_path.stat().st_size / 1e9:.1f} GB)")


def main():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True
    train_csv = DATA_ROOT / "annotations/train.csv"
    val_csv = DATA_ROOT / "annotations/val.csv"
    if not train_csv.is_file() or not val_csv.is_file():
        raise FileNotFoundError("Run the notebook's download cells first")
    if IMAGE_SIZE[0] % 14 or IMAGE_SIZE[1] % 14:
        raise ValueError("IMAGE_SIZE must be a multiple of the 14-pixel patch size")
    names, train_paths, train_labels = read_annotations(train_csv)
    val_names, val_paths, val_labels = read_annotations(val_csv)
    if names != val_names:
        raise ValueError("Train/validation attribute orders differ")
    for path in train_paths[:10] + val_paths[:10]:
        if not (DATA_ROOT / path).is_file():
            raise FileNotFoundError(DATA_ROOT / path)
    print(f"Train={len(train_paths):,}, validation={len(val_paths):,}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("No GPU found: feature extraction will run in float32 on the CPU and be very slow")
    # timm resamples the 448 x 448 position embedding to the pedestrian aspect ratio.
    trunk = timm.create_model(MODEL_ID, pretrained=USE_PRETRAINED_WEIGHTS, num_classes=0, img_size=IMAGE_SIZE)
    mean, std = trunk.pretrained_cfg["mean"], trunk.pretrained_cfg["std"]
    replace_rms_norms(trunk)
    encoder = RegionPooledEncoder(trunk).eval().requires_grad_(False)
    if device.type == "cuda":
        encoder.half()
    encoder.to(device)
    transform = image_transform(mean, std)
    train_features = extract_features(encoder, train_paths, transform, OUTPUT_ROOT / "upar_aimv2_3b_train_features.pt", device)
    val_features = extract_features(encoder, val_paths, transform, OUTPUT_ROOT / "upar_aimv2_3b_val_features.pt", device)
    head, score_method = fit_head(train_features, train_labels, val_features, val_labels, device)
    del train_features, val_features
    encoder.cpu()
    torch.cuda.empty_cache()
    package_submission(encoder, head, names, score_method, mean, std, transform,
                       DATA_ROOT / val_paths[0], val_labels[0])



## 3. Run validation and build the offline submission ZIP

This cell reports the full public validation retrieval metrics, selects NLL or L1 scoring on the fixed query subset, exports TorchScript in half precision, and checks the ZIP with a one-image inference call. The export traces the trunk on the CPU, which takes several minutes and about 11 GB of memory. Only the supplied training annotations are used for learning.


In [ ]:
main()

Download `upar_aimv2_3b_submission.zip` from Kaggle Outputs. Submit it only after comparing validation mADM and runtime with the DINOv3 result.
